# Phase 8b — Logit Lens: Reading Intermediate Layers with the Trained Classifier

Apply the **trained classification head** (BERT pooler + linear classifier) to
the `[CLS]` representation of every intermediate encoder layer, *without any
re-training*. The output is the prediction the model would emit if it had to
decide from that layer.

**Goal.** Distinguish three concepts that probing alone confounds:
1. **Information present** — measured by the linear probe (NB04).
2. **Information readable** — measured by the logit lens (this notebook).
3. **Components sufficient** — measured by activation patching (NB05).

**Pipeline.**
- **Setup** — paths, imports, helpers.
- **Part A** — load model, build per-emotion balanced subsample.
- **Part B** — extract `[CLS]` hidden states + apply pooler/classifier per layer.
- **Part C** — aggregate per layer, per group, per emotion (brechas).
- **Part D** — export every result as CSV under `results/notebook8b/`.

**Output CSVs** (consumed by Cap 5 §5.2):
- `logit_lens_per_layer.csv` — aggregate over the full 2 300 sample.
- `logit_lens_per_group.csv` — per crystallization group (early / middle / late).
- `logit_lens_per_emotion.csv` — per individual emotion.
- `probe_vs_lens_brechas.csv` — gap between probe F1 and gold sigmoid.

All computations use the 23 active emotions (5 excluded from GoEmotions).

## Setup


In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification

from src.data import load_goemotions

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
EXPORT_DIR  = os.path.join(RESULTS_DIR, 'notebook8b')
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(RESULTS_DIR, 'bert-goemotions-23emo-final')

EXCLUDED_EMOTIONS  = ['neutral', 'grief', 'nervousness', 'pride', 'relief']
N_ENCODER_LAYERS   = 12   # we ignore the embedding (logit lens on raw embeddings is meaningless)
SAMPLES_PER_EMOTION = 100  # 23 * 100 = 2 300 total examples (matches Cap 5 §5.2)
BATCH_SIZE         = 64
SEED               = 42

torch.manual_seed(SEED); np.random.seed(SEED)

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
print(f'Export dir:   {EXPORT_DIR}')

## Part A — Load Model, Test Set, Crystallization Groups

Pull the crystallization layer of every emotion from NB04 and use it to
partition the 23 emotions into three groups (early / middle / late). The
subsample is balanced: 100 examples per emotion (23 × 100 = 2 300).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification', output_hidden_states=True,
)
model.eval().to(DEVICE)

_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
num_labels = len(emotion_names)
emotion_to_idx = {e: i for i, e in enumerate(emotion_names)}

print(f'Model loaded from {MODEL_PATH}')
print(f'Test set: {len(test_ds)} samples, {num_labels} emotions')
print(f'Emotions: {emotion_names}')

In [ ]:
# Load crystallization groups from NB04 (fall back to flat results dir).
def _first_existing(*paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

crystal_path = _first_existing(
    os.path.join(RESULTS_DIR, 'notebook4', 'crystallization_layers.csv'),
    os.path.join(RESULTS_DIR, 'crystallization_layers.csv'),
)
if crystal_path is None:
    raise FileNotFoundError('crystallization_layers.csv not found. Run NB04 first.')

crystal_df = pd.read_csv(crystal_path)
print(f'Loaded crystallization layers from {crystal_path}')
print(f'Columns: {list(crystal_df.columns)}')

# NB04 stores crystallization_layer as 1-indexed (0 = embedding, 1 = L0, ..., 12 = L11).
# Group encoder layers L0..L11 (i.e. crystallization_layer 1..12) into three bands:
#   early  : L0..L3   -> crystallization_layer in {1, 2, 3, 4}
#   middle : L4..L7   -> crystallization_layer in {5, 6, 7, 8}
#   late   : L8..L11  -> crystallization_layer in {9, 10, 11, 12}
def emotion_group(layer):
    if pd.isna(layer): return 'middle'
    if layer <= 4:    return 'early'
    if layer <= 8:    return 'middle'
    return 'late'

crystal_df['group'] = crystal_df['crystallization_layer'].apply(emotion_group)
emotion_to_group = dict(zip(crystal_df['emotion'], crystal_df['group']))

for g in ('early', 'middle', 'late'):
    members = sorted([e for e, gg in emotion_to_group.items() if gg == g])
    print(f'  {g:6s}: {len(members)} emotions -> {members}')

In [ ]:
# Per-emotion balanced subsample: SAMPLES_PER_EMOTION examples where the
# emotion is positively present in the multi-label vector. Each example is
# tagged with a 'target_emotion' so we know which probability to track later.
rng = np.random.default_rng(SEED)

subsample_rows = []
for emo_idx, emo in enumerate(emotion_names):
    candidate_indices = [i for i in range(len(test_ds))
                         if test_ds[i]['labels'][emo_idx] > 0.5]
    if len(candidate_indices) == 0:
        print(f'  WARNING: no test examples for {emo}, skipping')
        continue
    pick = rng.choice(candidate_indices,
                       size=min(SAMPLES_PER_EMOTION, len(candidate_indices)),
                       replace=False)
    for idx in pick:
        subsample_rows.append({
            'sample_idx': int(idx),
            'target_emotion': emo,
            'target_emotion_idx': emo_idx,
            'group': emotion_to_group.get(emo, 'middle'),
        })

subsample_df = pd.DataFrame(subsample_rows)
print(f'\nSubsample size: {len(subsample_df)} (target: {num_labels * SAMPLES_PER_EMOTION})')
for g in ('early', 'middle', 'late'):
    n_e = subsample_df[subsample_df['group'] == g]['target_emotion'].nunique()
    n_s = (subsample_df['group'] == g).sum()
    print(f'  {g:6s}: {n_e} emotions, {n_s} samples')

## Part B — Extract Hidden States and Apply the Classification Head

For every example in the subsample, extract the `[CLS]` hidden state at every
encoder layer (1..12, ignoring the embedding) and apply
$\hat{p}_\ell = \sigma(W_{\text{cls}}\cdot\tanh(W_{\text{pool}}\cdot h_\ell^{[\text{CLS}]} + b_{\text{pool}}) + b_{\text{cls}})$.
Note: the pooler and classifier were trained on layer 12 only, so absolute
magnitudes in earlier layers are not calibrated (Cap 5 §5.2 caveat).
Relative comparisons across layers are still informative.

In [ ]:
@torch.no_grad()
def logit_lens_for_subsample(model, dataset, subsample_df, data_collator,
                              batch_size=BATCH_SIZE):
    """Return tensor of shape (n_samples, n_encoder_layers, num_labels) with sigmoid probs."""
    model.eval()
    dev = next(model.parameters()).device
    pooler     = model.bert.pooler
    classifier = model.classifier

    sample_indices = subsample_df['sample_idx'].tolist()
    n_samples      = len(sample_indices)
    n_layers       = N_ENCODER_LAYERS
    num_labels     = model.config.num_labels

    sigmoids = np.zeros((n_samples, n_layers, num_labels), dtype=np.float32)
    t0 = time.time()
    n_batches = (n_samples + batch_size - 1) // batch_size

    for b_start in range(0, n_samples, batch_size):
        b_end   = min(b_start + batch_size, n_samples)
        idx_chunk = sample_indices[b_start:b_end]
        batch = data_collator([dataset[j] for j in idx_chunk])

        outputs = model(
            input_ids=batch['input_ids'].to(dev),
            attention_mask=batch['attention_mask'].to(dev),
            output_hidden_states=True,
        )
        # outputs.hidden_states is a tuple of length 13: [embedding, layer1, ..., layer12]
        for layer_idx in range(n_layers):
            hidden = outputs.hidden_states[layer_idx + 1]   # skip embedding
            cls    = hidden[:, 0, :]                          # [batch, hidden]
            pooled = pooler(hidden)                           # pooler reads [:, 0]
            logits = classifier(pooled)                       # [batch, num_labels]
            sigmoids[b_start:b_end, layer_idx, :] = torch.sigmoid(logits).cpu().numpy()

        if (b_start // batch_size + 1) % 10 == 0:
            done = b_start // batch_size + 1
            print(f'  Batch {done}/{n_batches} ({time.time()-t0:.0f}s)')

    print(f'Done in {time.time()-t0:.1f}s')
    return sigmoids   # (n_samples, n_layers, num_labels)


print(f'Running logit lens on {len(subsample_df)} samples x {N_ENCODER_LAYERS} layers...')
lens_sigmoids = logit_lens_for_subsample(model, test_ds, subsample_df, data_collator)
print(f'Sigmoids shape: {lens_sigmoids.shape}  (samples x layers x emotions)')

## Part C — Aggregations

Three aggregation views:
1. **Per layer (global)** — mean gold sigmoid, top-1 sigmoid, total mass over
   all 2 300 samples.
2. **Per group, per layer** — same metrics but stratified by crystallization
   group (used for Cap 5 Fig 5.3).
3. **Per emotion, per layer** — fine-grained, used to compute the brecha
   between probe F1 and gold sigmoid (Cap 5 Tab 5.4).

In [ ]:
# Per-sample stats across layers ------------------------------------------------
target_idx = subsample_df['target_emotion_idx'].to_numpy()
n_samples  = lens_sigmoids.shape[0]

# Gold sigmoid: probability assigned to the target emotion at each layer.
gold_per_sample_layer = lens_sigmoids[np.arange(n_samples)[:, None],
                                       np.arange(N_ENCODER_LAYERS)[None, :],
                                       target_idx[:, None]]   # (n_samples, n_layers)

# Top-1 sigmoid: max probability across all 23 emotions at each layer.
top1_per_sample_layer = lens_sigmoids.max(axis=2)              # (n_samples, n_layers)

# Total mass: sum of the 23 sigmoids at each layer.
mass_per_sample_layer = lens_sigmoids.sum(axis=2)              # (n_samples, n_layers)

print('Per-sample stats computed.')
print(f'  gold_per_sample_layer: {gold_per_sample_layer.shape}')
print(f'  top1_per_sample_layer: {top1_per_sample_layer.shape}')
print(f'  mass_per_sample_layer: {mass_per_sample_layer.shape}')

In [ ]:
# Aggregation 1: per layer (global) --------------------------------------------
per_layer_rows = []
for layer in range(N_ENCODER_LAYERS):
    per_layer_rows.append({
        'layer'     : layer + 1,            # 1..12 (encoder index, NOT including embedding)
        'layer_name': f'L{layer}',          # display name (L0..L11) consistent with the memoria
        'gold_sigmoid' : float(gold_per_sample_layer[:, layer].mean()),
        'top1_sigmoid' : float(top1_per_sample_layer[:, layer].mean()),
        'total_mass'   : float(mass_per_sample_layer[:, layer].mean()),
        'n_samples'    : n_samples,
    })
per_layer_df = pd.DataFrame(per_layer_rows)
per_layer_df.to_csv(os.path.join(EXPORT_DIR, 'logit_lens_per_layer.csv'), index=False)
print('Aggregation 1 (per layer, global):')
print(per_layer_df.to_string(index=False))

In [ ]:
# Aggregation 2: per group, per layer ------------------------------------------
groups       = subsample_df['group'].to_numpy()
per_group_rows = []
for g in ('early', 'middle', 'late'):
    mask = (groups == g)
    for layer in range(N_ENCODER_LAYERS):
        per_group_rows.append({
            'group'        : g,
            'layer'        : layer + 1,
            'layer_name'   : f'L{layer}',
            'gold_sigmoid' : float(gold_per_sample_layer[mask, layer].mean()),
            'top1_sigmoid' : float(top1_per_sample_layer[mask, layer].mean()),
            'total_mass'   : float(mass_per_sample_layer[mask, layer].mean()),
            'n_samples'    : int(mask.sum()),
        })
per_group_df = pd.DataFrame(per_group_rows)
per_group_df.to_csv(os.path.join(EXPORT_DIR, 'logit_lens_per_group.csv'), index=False)
print('Aggregation 2 (per group, per layer):')
for g in ('early', 'middle', 'late'):
    sub = per_group_df[per_group_df['group'] == g]
    print(f'\n--- {g} (n = {sub.iloc[0]["n_samples"]}) ---')
    print(sub[['layer_name', 'gold_sigmoid', 'top1_sigmoid', 'total_mass']].to_string(index=False))

In [ ]:
# Aggregation 3: per emotion, per layer ----------------------------------------
per_emotion_rows = []
for emo_idx, emo in enumerate(emotion_names):
    mask = (target_idx == emo_idx)
    if mask.sum() == 0:
        continue
    for layer in range(N_ENCODER_LAYERS):
        per_emotion_rows.append({
            'emotion'      : emo,
            'group'        : emotion_to_group.get(emo, 'middle'),
            'layer'        : layer + 1,
            'layer_name'   : f'L{layer}',
            'gold_sigmoid' : float(gold_per_sample_layer[mask, layer].mean()),
            'top1_sigmoid' : float(top1_per_sample_layer[mask, layer].mean()),
            'total_mass'   : float(mass_per_sample_layer[mask, layer].mean()),
            'n_samples'    : int(mask.sum()),
        })
per_emotion_df = pd.DataFrame(per_emotion_rows)
per_emotion_df.to_csv(os.path.join(EXPORT_DIR, 'logit_lens_per_emotion.csv'), index=False)
print(f'Aggregation 3 (per emotion, per layer): {len(per_emotion_df)} rows')
print(per_emotion_df.head(15).to_string(index=False))

In [ ]:
# Aggregation 4: brechas (probe F1 - gold sigmoid) per group per layer ---------
# Pull probe results from NB04. We accept either the long format
# (probe_results_long.csv with columns: emotion, layer_idx, layer_name, probe_f1)
# or the wide format (probe_results.csv with columns: emotion, Emb, L0..L11).
probe_long_path = _first_existing(
    os.path.join(RESULTS_DIR, 'notebook4', 'probe_results_long.csv'),
    os.path.join(RESULTS_DIR, 'probe_results_long.csv'),
)
probe_wide_path = _first_existing(
    os.path.join(RESULTS_DIR, 'notebook4', 'probe_results.csv'),
    os.path.join(RESULTS_DIR, 'probe_results.csv'),
)

probe_long = None
if probe_long_path is not None:
    probe_long = pd.read_csv(probe_long_path)
    print(f'Loaded probe results (long) from {probe_long_path}')
elif probe_wide_path is not None:
    probe_wide = pd.read_csv(probe_wide_path)
    print(f'Loaded probe results (wide) from {probe_wide_path}; melting to long format.')
    probe_long = probe_wide.melt(id_vars='emotion', var_name='layer_name', value_name='probe_f1')
else:
    print('WARNING: probe results not found, skipping brechas computation.')

if probe_long is None:
    brechas_df = None
else:
    # Standardise: keep only encoder layers L0..L11 (drop the embedding row).
    if 'layer_name' not in probe_long.columns and 'layer_idx' in probe_long.columns:
        probe_long['layer_name'] = probe_long['layer_idx'].apply(
            lambda l: 'Emb' if int(l) == 0 else f'L{int(l) - 1}'
        )
    if 'probe_f1' not in probe_long.columns and 'f1_score' in probe_long.columns:
        probe_long = probe_long.rename(columns={'f1_score': 'probe_f1'})
    probe_long = probe_long[probe_long['layer_name'] != 'Emb'].copy()
    probe_long['group'] = probe_long['emotion'].map(emotion_to_group)

    # Mean probe F1 per (group, layer)
    probe_per_group = (probe_long.groupby(['group', 'layer_name'])['probe_f1']
                                  .mean().reset_index())
    # Merge with gold sigmoid per group
    lens_per_group = per_group_df[['group', 'layer_name', 'gold_sigmoid']]
    brechas_df = probe_per_group.merge(lens_per_group, on=['group', 'layer_name'])
    brechas_df['brecha'] = brechas_df['probe_f1'] - brechas_df['gold_sigmoid']
    brechas_df.to_csv(os.path.join(EXPORT_DIR, 'probe_vs_lens_brechas.csv'), index=False)

    print('Aggregation 4 (brechas):')
    pivot = brechas_df.pivot(index='group', columns='layer_name', values='brecha').round(3)
    cols = [f'L{i}' for i in range(N_ENCODER_LAYERS) if f'L{i}' in pivot.columns]
    print(pivot[cols])
    print('\nMax brecha per group:')
    for g in ('early', 'middle', 'late'):
        sub = brechas_df[brechas_df['group'] == g]
        if len(sub) > 0:
            best = sub.loc[sub['brecha'].idxmax()]
            print(f'  {g:6s}: {best["brecha"]:.3f} at {best["layer_name"]}')

## Part D — Quick Visualisations

Two figures: aggregate U pattern and per-group breakdown. Both are also saved
to disk so they can be reused if needed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = per_layer_df['layer_name']
ax.plot(x, per_layer_df['top1_sigmoid'], 'o-', label='Top-1 sigmoid', color='steelblue')
ax.plot(x, per_layer_df['gold_sigmoid'], 's-', label='Gold sigmoid',  color='crimson')
ax.plot(x, per_layer_df['total_mass']/10, '^--', label='Total mass / 10', color='gray')
ax.set_xlabel('Encoder layer (CLS)')
ax.set_ylabel('Probability (mean)')
ax.set_title('Logit lens — aggregate U pattern')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(EXPORT_DIR, 'logit_lens_aggregate_u.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {'early': 'tab:blue', 'middle': 'tab:orange', 'late': 'tab:red'}
for g in ('early', 'middle', 'late'):
    sub = per_group_df[per_group_df['group'] == g]
    ax.plot(sub['layer_name'], sub['gold_sigmoid'], 'o-',
            color=colors[g], label=f'{g} ({sub.iloc[0]["n_samples"]} samples)')
ax.set_xlabel('Encoder layer (CLS)')
ax.set_ylabel('Gold sigmoid (mean)')
ax.set_title('Logit lens by crystallization group')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(EXPORT_DIR, 'logit_lens_by_group.png'), dpi=200, bbox_inches='tight')
plt.show()

## Summary

Outputs in `results/notebook8b/`:
- `logit_lens_per_layer.csv` — aggregate U pattern (12 rows).
- `logit_lens_per_group.csv` — per crystallization group × layer (36 rows).
- `logit_lens_per_emotion.csv` — per emotion × layer (276 rows).
- `probe_vs_lens_brechas.csv` — gap between probe F1 and gold sigmoid.
- `logit_lens_aggregate_u.png`, `logit_lens_by_group.png` — visualisations.

**How to validate against the memoria.** Compare:
- `per_layer_df['gold_sigmoid']` at L7 vs Cap 5 §5.2 ($\approx$ 0,02).
- `per_layer_df['gold_sigmoid']` at L11 vs Cap 5 §5.2 ($\approx$ 0,44).
- `brechas_df` max per group vs Cap 5 Tab 5.4 (0,63 / 0,52 / 0,44).

If numbers diverge by more than rounding, update Cap 5 §5.2 with the
freshly-computed values.